In [ ]:
!pip install pycirclize

In [ ]:
from pycirclize import Circos
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# Load contacts file
df_links = pd.read_csv("Figure_5A_contacts_interchain_merged_thr0.5.csv")

# Chain -> Protein mapping
chain_to_protein = {
    "t2": "TSC22D2",
    "w1": "WNK1",
    "n1": "NRBP1",
    "o": "OXSR1"
}

# Define sectors
sectors = {
    "TSC22D2": 780,
    "WNK1": 2382,
    "NRBP1": 535,
    "OXSR1": 527
}

# Protein colors
protein_colors = {
    "TSC22D2": "#ffe5b6",
    "WNK1": "#49c1bb",
    "NRBP1": "#bababa",
    "OXSR1": "#6dabc6"
}

# Color mixing function
def mix_colors(hex1, hex2):
    rgb1 = np.array(mcolors.to_rgb(hex1))
    rgb2 = np.array(mcolors.to_rgb(hex2))
    mixed = (rgb1 + rgb2) / 2
    return mcolors.to_hex(mixed)

# Circos base
circos = Circos(sectors, space=2)

# Add protein color track
for sector in circos.sectors:
    protein = sector.name
    track = sector.add_track((82, 92))
    track.axis(ec="none", fc="none")
    values = np.ones(sectors[protein])
    color = protein_colors.get(protein, "#cccccc")
    track.heatmap(values[np.newaxis, :], cmap=mcolors.ListedColormap([color]),
                  vmin=0, vmax=1, rect_kws=dict(ec="none"))

# Add motif track for TSC22D2 only
motifs = {
    "TSC22D2": [
        (7, 17,  "#38876f"), #FXV/I motif
        (161, 164, "#154ea0"), #Rphi1 or RFXV motif
        (173, 190, "#d8c625"), #Rphi2 or TbrN motif
    ],
    "WNK1": [
        (221, 479,  "#49c1bb"), #Kinase domain
        (481, 571,  "#49c1bb"), #CCTL1 domain
        (1116, 1195, "#49c1bb"), #CCTL2 domain
        (1257, 1260, "#49c1bb"), #RFXV motif
        (1859, 1862, "#49c1bb"), #Coiled-coil motif
    ],
    "NRBP1": [
        (68, 327, "#bababa"), #Kinase-like domain
        (464, 476, "#bababa"), #CCT domain
    ],
    "OXSR1": [
        (17, 291, "#6dabc6"), #Kinase domain
        (434, 491, "#6dabc6"), #CCT domain
    ]
}

# Add motif track for all proteins in the motif dictionary
for sector in circos.sectors:
    protein = sector.name
    if protein not in motifs:
        continue

    track = sector.add_track((95, 100))  # outermost radius
    track.axis(ec="none", fc="none")

    arr = np.zeros(sectors[protein])  # length = number of residues in this protein

    for (start, end, color) in motifs[protein]:
        arr[start-1:end] = 1  # highlight region
        cmap = mcolors.ListedColormap(["none", color])
        track.heatmap(
            arr[np.newaxis, :],
            cmap=cmap,
            vmin=0, vmax=1,
            rect_kws=dict(ec="none")
        )

# Add links with mixed colors
for _, row in df_links.iterrows():
    res1, chain1 = row["Residue 1"], row["Chain 1"]
    res2, chain2 = row["Residue 2"], row["Chain 2"]
    prob = row["Probability"]

    if chain1 not in chain_to_protein or chain2 not in chain_to_protein:
        continue
    prot1, prot2 = chain_to_protein[chain1], chain_to_protein[chain2]

    if prot1 == prot2:
        continue
         
    alpha = min(max(prob, 0.5), 1.0)

    color1 = protein_colors.get(prot1, "#b3b3b3")
    color2 = protein_colors.get(prot2, "#b3b3b3")
    line_color = color1 if color1 == color2 else mix_colors(color1, color2)

    circos.link(
        (prot1, int(res1), int(res1)),
        (prot2, int(res2), int(res2)),
        r1=77, r2=77,
        color=line_color,
        lw=0.1,
        alpha=alpha
    )

# Save
fig = circos.plotfig()
fig.savefig("Figure_5D_Binary_Interaction_T2_N1_O_W1.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig.savefig("Figure_5D_Binary_Interaction_T2_N1_O_W1.png", dpi=800, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
from pycirclize import Circos
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ===========================
# Load CSV
# ===========================
df_links = pd.read_csv("Figure_5C_Result_with_Category.csv")

# ===========================
# Chain → Protein mapping
# ===========================
chain_to_protein = {
    "t2": "TSC22D2",
    "w1": "WNK1",
    "n1": "NRBP1",
    "o": "OXSR1"
}

# ===========================
# Sector definition
# ===========================
sectors = {
    "TSC22D2": 780,
    "WNK1": 2382,
    "NRBP1": 535,
    "OXSR1": 527
}

# ===========================
# Base protein colors
# ===========================
protein_colors = {
    "TSC22D2": "#ffe5b6",
    "WNK1": "#49c1bb",
    "NRBP1": "#bababa",
    "OXSR1": "#6dabc6"
}

# ===========================
# Color mixing function
# ===========================
def mix_colors(hex1, hex2):
    rgb1 = np.array(mcolors.to_rgb(hex1))
    rgb2 = np.array(mcolors.to_rgb(hex2))
    mixed = (rgb1 + rgb2) / 2
    return mcolors.to_hex(mixed)

# ===========================
# Motif dictionary
# ===========================
motifs = {
    "TSC22D2": [
        (7, 17,  "#38876f"),
        (161, 164, "#154ea0"),
        (173, 190, "#d8c625"),
        (694, 749, "#d8c625"),
    ],
    "WNK1": [
        (221, 479,  "#49c1bb"),
        (481, 571,  "#49c1bb"),
        (1116, 1195, "#49c1bb"),
        (1257, 1260, "#49c1bb"),
        (1655, 1662, "#49c1bb"),
        (2246, 2257, "#49c1bb"),
    ],
    "NRBP1": [
        (68, 327, "#bababa"),
        (464, 476, "#bababa"),
    ],
    "OXSR1": [
        (17, 291, "#6dabc6"),
        (434, 491, "#6dabc6"),
    ]
}

# ===========================================================
# Function: Build a Circos base (protein track + motif track)
# ===========================================================
def build_base_circos():
    circos = Circos(sectors, space=2)

    # Protein background track
    for sector in circos.sectors:
        protein = sector.name
        track = sector.add_track((82, 92))
        track.axis(ec="none", fc="none")

        values = np.ones(sectors[protein])
        color = protein_colors.get(protein)
        track.heatmap(values[np.newaxis, :],
                      cmap=mcolors.ListedColormap([color]),
                      rect_kws=dict(ec="none"))

    # Motif track
    for sector in circos.sectors:
        protein = sector.name
        if protein not in motifs:
            continue

        track = sector.add_track((94, 100))
        track.axis(ec="none", fc="none")

        arr = np.zeros(sectors[protein])

        for (start, end, color) in motifs[protein]:
            arr[start-1:end] = 1
            cmap = mcolors.ListedColormap(["none", color])
            track.heatmap(arr[np.newaxis, :], cmap=cmap,
                          vmin=0, vmax=1,
                          rect_kws=dict(ec="none"))

    return circos


# ===========================================================
# Function: Add links according to filtering rule
# ===========================================================
def add_links(circos, df_filtered):
    for _, row in df_filtered.iterrows():
        res1, chain1 = row["Residue 1"], row["Chain 1"]
        res2, chain2 = row["Residue 2"], row["Chain 2"]

        # Skip non-mapped chains
        if chain1 not in chain_to_protein or chain2 not in chain_to_protein:
            continue

        prot1 = chain_to_protein[chain1]
        prot2 = chain_to_protein[chain2]

        # Skip self interaction
        if prot1 == prot2:
            continue

        # Mixing color rules
        color1 = protein_colors.get(prot1)
        color2 = protein_colors.get(prot2)
        line_color = mix_colors(color1, color2) if prot1 != prot2 else color1

        circos.link(
            (prot1, int(res1), int(res1)),
            (prot2, int(res2), int(res2)),
            r1=77, r2=77,
            lw=0.3,
            color=line_color,
            alpha=0.9
        )


# ===========================================================
# VERSION A — Only in File 1
# ===========================================================
df_only_file1 = df_links[df_links["Category"] == "Only_in_File1"]

circos_v1 = build_base_circos()
add_links(circos_v1, df_only_file1)

fig1 = circos_v1.plotfig()
fig1.savefig("Circos_Only_File1.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig1.savefig("Circos_Only_File1.png", dpi=800, bbox_inches="tight", transparent=True)


# ===========================================================
# VERSION B — Intersection
# ===========================================================
df_other = df_links[df_links["Category"] == "Intersection"]

circos_v2 = build_base_circos()
add_links(circos_v2, df_other)

fig2 = circos_v2.plotfig()
fig2.savefig("Circos_Intersection.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig2.savefig("Circos_Intersection.png", dpi=800, bbox_inches="tight", transparent=True)


# ===========================================================
# VERSION C — Only in File 2
# ===========================================================
df_other = df_links[df_links["Category"] == "Only_in_File2"]

circos_v2 = build_base_circos()
add_links(circos_v2, df_other)

fig2 = circos_v2.plotfig()
fig2.savefig("Circos_Only_File2.pdf", dpi=600, bbox_inches="tight", transparent=True)
fig2.savefig("Circos_Only_File2.png", dpi=800, bbox_inches="tight", transparent=True)



plt.show()
